## 📑 Core Retail Rest
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQCFGQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if run_date:
    # run_date provided: use it exactly as passed in (no max business date)
    pass
else:
    # run_date empty: fall back to max business date
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d')

# business_date = last WORKING day of the PREVIOUS month, taken directly from
# ref_calendar.PreviousMonthLastWorkingDate (holiday/weekend aware, year-boundary safe).
# CalendarDate is a string column, so run_date is compared as a string.
# e.g. run_date='2025-12-31' -> '2025-11-28'; run_date='2026-01-15' -> '2025-12-31'
_cal = spark.sql(f'''
    SELECT PreviousMonthLastWorkingDate AS LastWorkingDayPrevMonth
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE CalendarDate = '{run_date}'
''').collect()

if not _cal or _cal[0]['LastWorkingDayPrevMonth'] is None:
    raise ValueError(f"No PreviousMonthLastWorkingDate found in ref_calendar for run_date={run_date}")

business_date = str(_cal[0]['LastWorkingDayPrevMonth'])   # e.g. '2025-11-28'
# exprt_month keeps the YYYYMM format, derived from the previous month's last working day
exprt_month = business_date[:7].replace('-', '')          # e.g. '202511'

# SQL-literal form for the EXPRT_MONTH filters downstream
exprt_month_sql = f"'{exprt_month}'"

###### Schema of the table

In [0]:
# Define the schema name
schema_name = f"`{env_prefix}-cdh-delivery`.repoint"

# Create the schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

table_name = 'Core_RetailRest' 

schema = (
"BusinessDate date,"
"EXPRT_MONTH string,"
"CASE_CODE string,"
"CIF string,"
"AFM string,"
"CLEAN_ACCOUNT_NUMBER string,"
"RELATION_TYPE string,"
"CUST_FIRST_NAME string,"
"CUST_LAST_NAME string,"
"CUST_FATH_NAME string,"
"MOTHERS_NAME string,"
"CUST_BIRTHDATE string,"
"CUST_DECEASED string,"
"CUST_ID_TYPE string,"
"CUST_ID_NUM string,"
"STAFF_INDICATOR string,"
"ACCOUNT_NUMBER string,"
"MASTER_ACCOUNT string,"
"CASE_CURRENCY string,"
"CONTRACT_NO string,"
"APL_REF string,"
"CURRENT_BUCKET string,"
"CASE_OPEN_DATE string,"
"CASE_EXP_DATE string,"
"BILLING_DATE string,"
"DUE_DATE string,"
"DPD string,"
"CURRENT_DOSI decimal(19,2),"
"OUTSTANDING_BALANCE decimal(19,2),"
"TOTAL_DUE decimal(19,2),"
"MIN_PAY_DUE decimal(19,2),"
"FROZEN decimal(19,2),"
"MLT_AMOUNT decimal(19,2),"
"WOFF_AMOUNT decimal(19,2),"
"WOFF_DATE string,"
"LAST_PAY_AMOUNT decimal(19,2),"
"LAST_PAY_DATE string,"
"ACCOUNT_STATUS string,"
"ACCOUNT_STATUS_DATE string,"
"INTEREST_RATE string,"
"OBJECTIVE_STATUS string,"
"OBJECTIVE_STATUS_DATE string,"
"AGENCY_CODE string,"
"ASSIGN_DATE string,"
"PACKET_PROCESS_TYPE string,"
"QUE_DESC string,"
"QUEUE_TYPE string,"
"BU string,"
"HOST_BU string,"
"CASE_SOURCE string,"
"MENU_CODE string,"
"SOURCE_CODE string,"
"PRODUCT_CODE string,"
"PRODUCT_DESC string,"
"SUBPRODUCT_CODE string,"
"SUBPRODUCT_DESC string,"
"CREDIT_LIMIT decimal(19,2),"
"CUS_LPS string,"
"DEBT_LPS string,"
"DEBT_LPS_DATE string,"
"SECURITIZED_FLAG string,"
"SPECIAL_TERATMENT string,"
"LEGAL_ENTITY string,"
"BRANCH_CODE string,"
"BRANCH_DESC string,"
"COURT_DATE string,"
"PRE_COURT_DATE string,"
"CASE_CLOSE_DATE string,"
"SERVICING_TYPE string,"
"PREVIOUS_EXTERNAL_AGENCY string,"
"COLLECTOR_ID string,"
"PREVIOUS_PACKET_PROCESS_TYPE string,"
"GUARANTEE_INDICATOR string,"
"GUARANTEE_ORGANIZATION string,"
"GUARANTEE_ORGANIZATION_DESC string,"
"SUBSIDY_INDEX string,"
"CONNECTED_BORROWERS string,"
"LOAN_DURATION string,"
"ON_BALANCE decimal(19,2),"
"OFF_BALANCE decimal(19,2),"
"TOTAL_BALANCE decimal(19,2),"
"TOTAL_BALANCE_EMLT decimal(19,2),"
"SETTLEMENT_TYPE string,"
"SETTLEMENT_CURRENT_STATUS string,"
"QCCustomerL3869Status string,"
"InterbankStatusL3869 string,"
"Customer_Watchlist string,"
"CustomerL3869 string,"
"CourtDate string,"
"ApplicationSubmissionDate string,"
"SPV_DESC string,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### CL_FullPortfolioRest

In [0]:
CL_FullPortfolioRest = spark.sql(
f'''

select *, 1 as IsCard from `{env_prefix}-cdh-delivery`.repoint.cl_fullportfoliorest where BusinessDate = (SELECT MAX(BusinessDate) --change
                        FROM `{env_prefix}-cdh-delivery`.repoint.cl_fullportfoliorest --change
                        WHERE EXPRT_MONTH={exprt_month_sql}
                           )
'''
)
CL_FullPortfolioRest.createOrReplaceTempView("CL_FullPortfolioRest")

##### ML_FullPortfolioRest

In [0]:
ML_FullPortfolioRest = spark.sql(
f'''
select *, 1 as IsCard from `{env_prefix}-cdh-delivery`.repoint.ml_fullportfoliorest 
where BusinessDate = (SELECT MAX(BusinessDate)
                        FROM `{env_prefix}-cdh-delivery`.repoint.ml_fullportfoliorest 
                        WHERE EXPRT_MONTH={exprt_month_sql}
                           )
'''
)
ML_FullPortfolioRest.createOrReplaceTempView("ML_FullPortfolioRest")

##### SB_FullPortfolioRest

In [0]:
SB_FullPortfolioRest = spark.sql(
f'''
select *, 0 as IsCard from `{env_prefix}-cdh-delivery`.repoint.sb_fullportfoliorest 
where BusinessDate = (SELECT MAX(BusinessDate)
                      FROM `{env_prefix}-cdh-delivery`.repoint.ml_fullportfoliorest 
                        WHERE EXPRT_MONTH={exprt_month_sql}
                      )

'''
)
SB_FullPortfolioRest.createOrReplaceTempView("SB_FullPortfolioRest")

##### FullPortFolio

In [0]:
FullPortfolio = spark.sql(
f'''
SELECT *, left(ACCOUNT_NUMBER, charindex('-', ACCOUNT_NUMBER)-1) as AccountKeyForJoin
	FROM (
	      SELECT * 
          FROM CL_FullPortfolioRest
          UNION ALL
          SELECT * 
          FROM ML_FullPortfolioRest
          UNION ALL
          SELECT * 
          FROM SB_FullPortfolioRest
		  ) AS A
'''
)
FullPortfolio.createOrReplaceTempView("FullPortfolio")

##### Daily Portfolio

In [0]:
DailyPortfolioCore = spark.sql(
f'''
select dp.account_number, dp.spv_desc 
from `{env_prefix}-cdh`.landed.repointqcab_daily_portfolio dp  ---change
 where dp.BusinessDate = '{business_date}'
'''
)   
DailyPortfolioCore.createOrReplaceTempView("DailyPortfolioCore")


##### CorePerimeter Exposure

In [0]:
CorePerimeterExposure = spark.sql(
f'''
select distinct exposurekey
from `{env_prefix}-cdh`.primary.exposure dp
 where dp.BusinessDate = '{business_date}'
and portfoliokey = 2
and isactiveflag = true
'''
)   
CorePerimeterExposure.createOrReplaceTempView("CorePerimeterExposure")

##### Final_V0

In [0]:
Final_V0 = spark.sql(
f'''
SELECT a.* 
	      , b.SPV_DESC	
FROM FullPortfolio a
LEFT join DailyPortfolioCore b on a.AccountKeyForJoin = b.ACCOUNT_NUMBER
INNER join CorePerimeterExposure C on a.AccountKeyForJoin = C.EXPOSUREKEY
'''
)
Final_V0.createOrReplaceTempView("Final_V0")

##### CoreRetailRest

In [0]:
RA_CoreRetailRest = spark.sql(
f'''
SELECT
	     CAST('{run_date}' AS DATE) as BusinessDate
		,EXPRT_MONTH
		,CASE_CODE
		,CIF
		,AFM
		,CLEAN_ACCOUNT_NUMBER
		,RELATION_TYPE
		,CUST_FIRST_NAME
		,CUST_LAST_NAME
		,CUST_FATH_NAME
		,MOTHERS_NAME
		,date_format(try_to_date(CUST_BIRTHDATE),'yyyy-MM-dd') AS CUST_BIRTHDATE 
		,CUST_DECEASED
		,CUST_ID_TYPE
		,CUST_ID_NUM
		,STAFF_INDICATOR
		,ACCOUNT_NUMBER
		,MASTER_ACCOUNT
		,CASE_CURRENCY
		,CONTRACT_NO
		,APL_REF
		,CURRENT_BUCKET
		,date_format(try_to_date(CASE_OPEN_DATE),'yyyy-MM-dd') AS CASE_OPEN_DATE
	   ,date_format(try_to_date(CASE_EXP_DATE),'yyyy-MM-dd') AS CASE_EXP_DATE
	   ,date_format(try_to_date(BILLING_DATE),'yyyy-MM-dd') AS BILLING_DATE
	   ,date_format(try_to_date(DUE_DATE),'yyyy-MM-dd') AS DUE_DATE
		,DPD
		,cast(CURRENT_DOSI as decimal(19,2)) As CURRENT_DOSI
	   ,cast(OUTSTANDING_BALANCE as decimal(19,2)) As OUTSTANDING_BALANCE
	   ,cast(TOTAL_DUE as decimal(19,2)) As TOTAL_DUE
	   ,cast(MIN_PAY_DUE as decimal(19,2)) As MIN_PAY_DUE
	   ,cast(FROZEN as decimal(19,2)) As FROZEN
	   ,cast(MLT_AMOUNT as decimal(19,2)) As MLT_AMOUNT
	   ,cast(WOFF_AMOUNT as decimal(19,2)) As WOFF_AMOUNT
	   ,CASE WHEN WOFF_DATE = '-100' then null else date_format(try_to_date(WOFF_DATE),'yyyy-MM-dd') END AS WOFF_DATE
	   ,cast(LAST_PAY_AMOUNT as decimal(19,2)) As LAST_PAY_AMOUNT
	   ,date_format(try_to_date(LAST_PAY_DATE),'yyyy-MM-dd') AS LAST_PAY_DATE
	   ,ACCOUNT_STATUS
	   , case when ACCOUNT_STATUS_DATE = '-100' then null else date_format(try_to_date(ACCOUNT_STATUS_DATE),'yyyy-MM-dd') END AS ACCOUNT_STATUS_DATE
		,INTEREST_RATE
		,OBJECTIVE_STATUS
		,date_format(try_to_date(OBJECTIVE_STATUS_DATE),'yyyy-MM-dd') AS OBJECTIVE_STATUS_DATE
	   ,AGENCY_CODE
	   ,date_format(try_to_date(ASSIGN_DATE),'yyyy-MM-dd') AS ASSIGN_DATE
		,PACKET_PROCESS_TYPE
		,QUE_DESC
		,QUEUE_TYPE
		,BU
		,HOST_BU
		,CASE_SOURCE
		,MENU_CODE
		,SOURCE_CODE
		,PRODUCT_CODE
		,PRODUCT_DESC
		,SUBPRODUCT_CODE
		,SUBPRODUCT_DESC
		,cast(CREDIT_LIMIT as decimal(19,2)) As CREDIT_LIMIT
		,CUS_LPS
		,DEBT_LPS
		,date_format(try_to_date(DEBT_LPS_DATE),'yyyy-MM-dd') AS DEBT_LPS_DATE
		,SECURITIZED_FLAG
		,SPECIAL_TERATMENT
		,LEGAL_ENTITY
		,BRANCH_CODE
		,BRANCH_DESC
		,date_format(try_to_date(COURT_DATE),'yyyy-MM-dd') AS COURT_DATE
	   ,date_format(try_to_date(PRE_COURT_DATE),'yyyy-MM-dd') AS PRE_COURT_DATE
	   ,date_format(try_to_date(CASE_CLOSE_DATE),'yyyy-MM-dd') AS CASE_CLOSE_DATE
		,SERVICING_TYPE
		,PREVIOUS_EXTERNAL_AGENCY
		,COLLECTOR_ID
		,PREVIOUS_PACKET_PROCESS_TYPE
		,GUARANTEE_INDICATOR
		,GUARANTEE_ORGANIZATION
		,GUARANTEE_ORGANIZATION_DESC
		,SUBSIDY_INDEX
		,CONNECTED_BORROWERS
		,LOAN_DURATION
		,cast(ON_BALANCE as decimal(19,2)) As ON_BALANCE
	   ,cast(OFF_BALANCE as decimal(19,2)) As OFF_BALANCE
	   ,cast(TOTAL_BALANCE as decimal(19,2)) As TOTAL_BALANCE
	   ,cast(TOTAL_BALANCE_EMLT as decimal(19,2)) As TOTAL_BALANCE_EMLT
		,SETTLEMENT_TYPE
		,SETTLEMENT_CURRENT_STATUS
		,QCCustomerL3869Status       AS QCCustomerL3869Status
		,InterbankStatusL3869   AS InterbankStatusL3869
		,Customer_Watchlist AS Customer_Watchlist
		,CustomerL3869                AS CustomerL3869
		,date_format(try_to_date(CourtDate),'yyyy-MM-dd') AS CourtDate
	   ,date_format(try_to_date(ApplicationSubmissionDate),'yyyy-MM-dd') AS ApplicationSubmissionDate
		,SPV_DESC
	 FROM Final_V0 
  where spV_desc  like '%GEMINI%' OR  spV_desc = '' OR  spV_desc IS  NULL OR SPV_DESC = ' '
'''
)   
RA_CoreRetailRest.createOrReplaceTempView("RA_CoreRetailRest")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST(BusinessDate AS DATE) AS BusinessDate
, CAST(EXPRT_MONTH AS string) AS EXPRT_MONTH
, CAST(CASE_CODE AS string) AS CASE_CODE
, CAST(CIF AS string) AS CIF
, CAST(AFM AS string) AS AFM
, CAST(CLEAN_ACCOUNT_NUMBER AS string) AS CLEAN_ACCOUNT_NUMBER
, CAST(RELATION_TYPE AS string) AS RELATION_TYPE
, CAST(CUST_FIRST_NAME AS string) AS CUST_FIRST_NAME
, CAST(CUST_LAST_NAME AS string) AS CUST_LAST_NAME
, CAST(CUST_FATH_NAME AS string) AS CUST_FATH_NAME
, CAST(MOTHERS_NAME AS string) AS MOTHERS_NAME
, CAST(CUST_BIRTHDATE AS string) AS CUST_BIRTHDATE
, CAST(CUST_DECEASED AS string) AS CUST_DECEASED
, CAST(CUST_ID_TYPE AS string) AS CUST_ID_TYPE
, CAST(CUST_ID_NUM AS string) AS CUST_ID_NUM
, CAST(STAFF_INDICATOR AS string) AS STAFF_INDICATOR
, CAST(ACCOUNT_NUMBER AS string) AS ACCOUNT_NUMBER
, CAST(MASTER_ACCOUNT AS string) AS MASTER_ACCOUNT
, CAST(CASE_CURRENCY AS string) AS CASE_CURRENCY
, CAST(CONTRACT_NO AS string) AS CONTRACT_NO
, CAST(APL_REF AS string) AS APL_REF
, CAST(CURRENT_BUCKET AS string) AS CURRENT_BUCKET
, CAST(CASE_OPEN_DATE AS string) AS CASE_OPEN_DATE
, CAST(CASE_EXP_DATE AS string) AS CASE_EXP_DATE
, CAST(BILLING_DATE AS string) AS BILLING_DATE
, CAST(DUE_DATE AS string) AS DUE_DATE
, CAST(DPD AS string) AS DPD
, CAST(CURRENT_DOSI AS decimal(19,2)) AS CURRENT_DOSI
, CAST(OUTSTANDING_BALANCE AS decimal(19,2)) AS OUTSTANDING_BALANCE
, CAST(TOTAL_DUE AS decimal(19,2)) AS TOTAL_DUE
, CAST(MIN_PAY_DUE AS decimal(19,2)) AS MIN_PAY_DUE
, CAST(FROZEN AS decimal(19,2)) AS FROZEN
, CAST(MLT_AMOUNT AS decimal(19,2)) AS MLT_AMOUNT
, CAST(WOFF_AMOUNT AS decimal(19,2)) AS WOFF_AMOUNT
, CAST(WOFF_DATE AS string) AS WOFF_DATE
, CAST(LAST_PAY_AMOUNT AS decimal(19,2)) AS LAST_PAY_AMOUNT
, CAST(LAST_PAY_DATE AS string) AS LAST_PAY_DATE
, CAST(ACCOUNT_STATUS AS string) AS ACCOUNT_STATUS
, CAST(ACCOUNT_STATUS_DATE AS string) AS ACCOUNT_STATUS_DATE
, CAST(INTEREST_RATE AS string) AS INTEREST_RATE
, CAST(OBJECTIVE_STATUS AS string) AS OBJECTIVE_STATUS
, CAST(OBJECTIVE_STATUS_DATE AS string) AS OBJECTIVE_STATUS_DATE
, CAST(AGENCY_CODE AS string) AS AGENCY_CODE
, CAST(ASSIGN_DATE AS string) AS ASSIGN_DATE
, CAST(PACKET_PROCESS_TYPE AS string) AS PACKET_PROCESS_TYPE
, CAST(QUE_DESC AS string) AS QUE_DESC
, CAST(QUEUE_TYPE AS string) AS QUEUE_TYPE
, CAST(BU AS string) AS BU
, CAST(HOST_BU AS string) AS HOST_BU
, CAST(CASE_SOURCE AS string) AS CASE_SOURCE
, CAST(MENU_CODE AS string) AS MENU_CODE
, CAST(SOURCE_CODE AS string) AS SOURCE_CODE
, CAST(PRODUCT_CODE AS string) AS PRODUCT_CODE
, CAST(PRODUCT_DESC AS string) AS PRODUCT_DESC
, CAST(SUBPRODUCT_CODE AS string) AS SUBPRODUCT_CODE
, CAST(SUBPRODUCT_DESC AS string) AS SUBPRODUCT_DESC
, CAST(CREDIT_LIMIT AS decimal(19,2)) AS CREDIT_LIMIT
, CAST(CUS_LPS AS string) AS CUS_LPS
, CAST(DEBT_LPS AS string) AS DEBT_LPS
, CAST(DEBT_LPS_DATE AS string) AS DEBT_LPS_DATE
, CAST(SECURITIZED_FLAG AS string) AS SECURITIZED_FLAG
, CAST(SPECIAL_TERATMENT AS string) AS SPECIAL_TERATMENT
, CAST(LEGAL_ENTITY AS string) AS LEGAL_ENTITY
, CAST(BRANCH_CODE AS string) AS BRANCH_CODE
, CAST(BRANCH_DESC AS string) AS BRANCH_DESC
, CAST(COURT_DATE AS string) AS COURT_DATE
, CAST(PRE_COURT_DATE AS string) AS PRE_COURT_DATE
, CAST(CASE_CLOSE_DATE AS string) AS CASE_CLOSE_DATE
, CAST(SERVICING_TYPE AS string) AS SERVICING_TYPE
, CAST(PREVIOUS_EXTERNAL_AGENCY AS string) AS PREVIOUS_EXTERNAL_AGENCY
, CAST(COLLECTOR_ID AS string) AS COLLECTOR_ID
, CAST(PREVIOUS_PACKET_PROCESS_TYPE AS string) AS PREVIOUS_PACKET_PROCESS_TYPE
, CAST(GUARANTEE_INDICATOR AS string) AS GUARANTEE_INDICATOR
, CAST(GUARANTEE_ORGANIZATION AS string) AS GUARANTEE_ORGANIZATION
, CAST(GUARANTEE_ORGANIZATION_DESC AS string) AS GUARANTEE_ORGANIZATION_DESC
, CAST(SUBSIDY_INDEX AS string) AS SUBSIDY_INDEX
, CAST(CONNECTED_BORROWERS AS string) AS CONNECTED_BORROWERS
, CAST(LOAN_DURATION AS string) AS LOAN_DURATION
, CAST(ON_BALANCE AS decimal(19,2)) AS ON_BALANCE
, CAST(OFF_BALANCE AS decimal(19,2)) AS OFF_BALANCE
, CAST(TOTAL_BALANCE AS decimal(19,2)) AS TOTAL_BALANCE
, CAST(TOTAL_BALANCE_EMLT AS decimal(19,2)) AS TOTAL_BALANCE_EMLT
, CAST(SETTLEMENT_TYPE AS string) AS SETTLEMENT_TYPE
, CAST(SETTLEMENT_CURRENT_STATUS AS string) AS SETTLEMENT_CURRENT_STATUS
, CAST(QCCustomerL3869Status AS string) AS QCCustomerL3869Status
, CAST(InterbankStatusL3869 AS string) AS InterbankStatusL3869
, CAST(Customer_Watchlist AS string) AS Customer_Watchlist
, CAST(CustomerL3869 AS string) AS CustomerL3869
, CAST(CourtDate AS string) AS CourtDate
, CAST(ApplicationSubmissionDate AS string) AS ApplicationSubmissionDate
, CAST(SPV_DESC AS string) AS SPV_DESC
, CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM RA_CoreRetailRest A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_RetailRest: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["CASE_CODE"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
RetailRest = spark.sql(
f'''
SELECT * 
FROM {delivery_table_path}
WHERE BusinessDate = {run_date!r}
'''
)
RetailRest.createOrReplaceTempView("RetailRest")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/RetailRest/"
file_name = "RetailRest_" +run_date+ ".csv"
RetailRest.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)